# Audio Segmentation & Static Masking Pipeline

This Colab provides an end-to-end pipeline for clustering audio segments into continuous transmissions, extracting them, and applying dynamic pink-noise masking over sensitive/unwanted regions (e.g., PII, unintelligible audio, DTMF tones) before exporting the final audio to GCS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/common/chirp_and_gemini_segment_masked_audio.ipynb)

### Core Functions

1. **Transmission Building (Clustering)**: Clusters speech segments into coherent, contiguous "transmissions" of activity based on a **configurable silence gap threshold (defaults to 0.5 seconds)**. Only blocks containing valid transcriptions are retained.
2. **Collision-Aware Padding**: Calculates safe start/end boundaries for each extracted transmission block. **Dynamic padding is applied (configurable up to a default of 0.5 seconds)** to avoid truncating words, while strictly preventing overlaps with adjacent speech blocks.
3. **Advanced Audio Masking**: Obfuscates non-speech or sensitive events (`PII`, `UNINTELLIGIBLE`, `DTMF`, `RINGING`) using dynamic pink noise generation:
   - **Transient Expansion**: Expands masks by a fixed 50ms to fully capture click/pop transients.
   - **Safe Clamping**: Ensures masking boundaries never bleed into actual transcribed speech.
   - **Seamless Crossfades**: Applies a fixed 10ms crossfade curve to the source audio and the generated pink noise to eliminate sudden auditory pops.
   - **Volume-Matched Pink Noise**: Measures local RMS power and injects volume-matched pink noise into the muted region for a natural audio transition.
4. **Automated Manifest & GCS Export**:
   - Writes the final processed audio as high-quality `.flac` files to GCS.
   - Exports a unified `batch_manifest.jsonl` mapping each segment to its metadata (GCS URI, ground truth text, offset, and duration) to facilitate downstream ASR evaluations.


In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    colorednoise

In [ ]:
# @title Imports
from collections import defaultdict
import json
from pathlib import Path
import sys
from urllib.parse import urlparse

import colorednoise as cn
from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
from loguru import logger
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.notebook import tqdm

In [ ]:
# @title Authentication and client initialization
auth.authenticate_user()


# User Configuration from Colab Secrets
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Pipeline Configuration

# fmt: off
# @markdown ### 1. GCS Input & Output Paths
# @markdown Path to the manifest file relative to the 'manifests/' directory (e.g. fire_notifications/eval/manifest.json)
MANIFEST_FILE_PATH = "echo/eval/manifest.json"  # @param {type:"string"}
# @markdown Path to the audio directory relative to the 'audio/' directory (e.g. fire_notifications/eval)
AUDIO_DIR_PATH = "echo/eval"  # @param {type:"string"}
# @markdown GCS path for output relative to the 'segmented_audio/' directory (e.g. fire_notifications_masked)
OUTPUT_PATH = "echo/eval_audio_masked"  # @param {type:"string"}
# Overwrite existing GCS files?
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

# @markdown ### 2. Segmentation Parameters
# Maximum gap allowed before splitting into a new transmission block (seconds)
TRANSMISSION_GAP_THRESHOLD = 0.5  # @param {type:"slider", min:0.1, max:3.0, step:0.1}
# Minimum gap required to micro-harvest pure background static (seconds)
STATIC_HARVEST_MIN = 0.5  # @param {type:"slider", min:0.1, max:3.0, step:0.1}
# Collision-Aware Padding (ie, max padding value)
DESIRED_PAD = 0.5  # @param {type:"slider", min:0.0, max:2.0, step:0.1}

# @markdown ### 3. System Settings
LOCAL_BASE_PATH = "/content"  # @param {type:"string"}
LOG_LEVEL = "INFO"  # @param ["DEBUG", "INFO", "WARNING", "ERROR"]
# fmt: on

assert MANIFEST_FILE_PATH, "MANIFEST_FILE_PATH must be provided."
assert AUDIO_DIR_PATH, "AUDIO_DIR_PATH must be provided."
assert OUTPUT_PATH, "OUTPUT_PATH must be provided."

CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segmented_audio_masked"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(
    sys.stderr, level=LOG_LEVEL, format="<level>{level}</level>: {message}"
)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# @title Transmission Building & Static Masking Logic
def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Downloads raw audio to local cache and validates file integrity."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")
    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    # If file exists but is 0 bytes, it's corrupt. Delete it.
    if local_path.exists() and local_path.stat().st_size == 0:
        logger.warning(
            f"Found empty file {filename}, removing for re-download..."
        )
        local_path.unlink()

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        try:
            gcs_client.bucket(bucket_name).blob(blob_name).download_to_filename(
                str(local_path)
            )
        except Exception as e:
            logger.error(f"Failed to download {filename} from GCS: {e}")
            raise e
    return str(local_path)


def cleanup_gcs_output(bucket_name: str, prefix: str) -> None:
    """Deletes the GCS 'directory' (prefix) and all contents recursively."""
    bucket = gcs_client.bucket(bucket_name)
    blobs = list(bucket.list_blobs(prefix=prefix))

    if blobs:
        logger.info(
            f"Deleting prefix and all contents: gs://{bucket_name}/{prefix} ({len(blobs)} files)..."
        )
        bucket.delete_blobs(blobs)
        logger.info("Cleanup complete.")
    else:
        logger.info(
            f"Target prefix gs://{bucket_name}/{prefix} is already empty."
        )


def build_transmissions(
    anchor_segments: list[dict], threshold: float
) -> list[list[dict]]:
    """Clusters ONLY speech segments into dense transmissions of activity."""
    if not anchor_segments:
        return []

    transmissions = []
    current_transmission = [anchor_segments[0]]
    max_end = anchor_segments[0]["offset"] + anchor_segments[0]["duration"]

    for curr_seg in anchor_segments[1:]:
        gap = curr_seg["offset"] - max_end

        if gap <= threshold:
            current_transmission.append(curr_seg)
            max_end = max(max_end, curr_seg["offset"] + curr_seg["duration"])
        else:
            transmissions.append(current_transmission)
            current_transmission = [curr_seg]
            max_end = curr_seg["offset"] + curr_seg["duration"]

    transmissions.append(current_transmission)
    return transmissions

In [ ]:
# @title Execute Segmentation & Masking Pipeline
def generate_pink_noise(samples: int, volume_scale: float = 0.1) -> np.ndarray:
    """Generates pink noise using the specialized colorednoise library."""
    if samples <= 0:
        return np.array([], dtype=np.float32)
    # beta=1 for pink noise (1/f)
    pink_noise = cn.powerlaw_psd_gaussian(1, samples)

    # 1. Remove DC offset (Center the waveform at zero)
    pink_noise = pink_noise - np.mean(pink_noise)

    # 2. Normalize and scale
    if np.max(np.abs(pink_noise)) > 0:
        pink_noise = (pink_noise / np.max(np.abs(pink_noise))) * volume_scale
    return pink_noise.astype(np.float32)


def run_segmentation_pipeline() -> None:
    full_output_prefix = f"segmented_audio/{OUTPUT_PATH.strip('/')}"

    # Using existing constants directly
    if OVERWRITE_EXISTING:
        cleanup_gcs_output(GCS_BUCKET, full_output_prefix)

    output_bucket = gcs_client.bucket(GCS_BUCKET)

    full_manifest_path = f"manifests/{MANIFEST_FILE_PATH.lstrip('/')}"
    m_blob = output_bucket.blob(full_manifest_path)
    content = m_blob.download_as_text()

    if not content or len(content.strip()) < 2:
        logger.error(f"Manifest at {full_manifest_path} appears to be empty.")
        return

    try:
        manifest_data = json.loads(content)
    except json.JSONDecodeError as e:
        logger.error(f"Failed to parse manifest: {e}")
        return

    files_to_process = defaultdict(list)
    for entry in manifest_data:
        files_to_process[entry["audio_filepath"]].append(entry)

    final_manifest_entries = []

    for json_audio_path, raw_segments in tqdm(
        files_to_process.items(), desc="Processing Audio Files"
    ):
        filename = Path(json_audio_path).name
        example_id = Path(filename).stem
        # Use flexible audio path structure relative to the bucket and audio directory
        true_gcs_uri = (
            f"gs://{GCS_BUCKET}/audio/{AUDIO_DIR_PATH.strip('/')}/{filename}"
        )

        raw_segments.sort(key=lambda x: x["offset"])
        local_src_path = ensure_local_gcs_audio(true_gcs_uri)

        y, sr = sf.read(local_src_path)
        if y.ndim > 1:  # Ensure mono
            y = y.mean(axis=1)

        logger.info(f"Processing {example_id}...")

        human_categories = {"TRANSCRIPTION", "PII", "UNINTELLIGIBLE"}
        human_segments_only = [
            seg
            for x, seg in enumerate(raw_segments)
            if seg.get("category") in human_categories
        ]

        # Cluster segments into harvested blocks
        clustered_segments = build_transmissions(
            human_segments_only, TRANSMISSION_GAP_THRESHOLD
        )

        # Only keep blocks that contain actual transcription text
        final_segments = [
            block
            for block in clustered_segments
            if any(seg.get("category") == "TRANSCRIPTION" for seg in block)
        ]

        max_audio_time = len(y) / sr

        for i, segment_block in enumerate(final_segments):
            # 1. Extract Ground Truth text
            gt_texts = [
                seg.get("text", "").strip()
                for seg in segment_block
                if seg.get("category") == "TRANSCRIPTION" and seg.get("text")
            ]
            segment_gt_text = " ".join(gt_texts)

            # 2. Identify the core speech boundaries (ignore leading/trailing noise/PII)
            transcription_segs = [
                s for s in segment_block if s.get("category") == "TRANSCRIPTION"
            ]
            core_start = min(s["offset"] for s in transcription_segs)
            core_end = max(
                s["offset"] + s["duration"] for s in transcription_segs
            )

            # Only check for collisions against other TRANSCRIPTION segments in the full file
            transcription_only_segments = [
                s for s in raw_segments if s.get("category") == "TRANSCRIPTION"
            ]

            # Find closest preceding TRANSCRIPTION label
            preceding_segs = [
                s
                for s in transcription_only_segments
                if s["offset"] + s["duration"] <= core_start
            ]
            available_front_gap = (
                core_start
                - max(s["offset"] + s["duration"] for s in preceding_segs)
                if preceding_segs
                else DESIRED_PAD
            )
            safe_front_pad = max(0.0, min(DESIRED_PAD, available_front_gap))

            # Find closest following TRANSCRIPTION label
            following_segs = [
                s
                for s in transcription_only_segments
                if s["offset"] >= core_end
            ]
            available_back_gap = (
                min(s["offset"] for s in following_segs) - core_end
                if following_segs
                else DESIRED_PAD
            )
            safe_back_pad = max(0.0, min(DESIRED_PAD, available_back_gap))

            # 3. Final Audio Bounds
            segment_start = max(0.0, core_start - safe_front_pad)
            segment_end = min(max_audio_time, core_end + safe_back_pad)

            segment_id = f"{i:03d}"

            out_filename = f"/content/segmented_audio_masked/{example_id}__seg{segment_id}.flac"
            blob_name = (
                f"{full_output_prefix}/{example_id}/{Path(out_filename).name}"
            )
            output_blob = output_bucket.blob(blob_name)

            if not OVERWRITE_EXISTING and output_blob.exists():
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "text": segment_gt_text,
                        "example_id": example_id,
                        "segment_id": segment_id,
                        "offset": segment_start,
                        "duration": segment_end - segment_start,
                    }
                )
                continue

            start_samp = int(segment_start * sr)
            end_samp = int(segment_end * sr)
            segment_audio = y[start_samp:end_samp].copy()

            categories_to_mask = {"PII", "UNINTELLIGIBLE", "DTMF", "RINGING"}
            for seg in (
                raw_segments
            ):  # Note: Ensure you iterate over raw_segments here
                cat = seg.get("category")
                if cat in categories_to_mask:
                    # Expand mask by 50ms to swallow unannotated transients/mic clicks
                    mask_start = seg["offset"] - 0.05
                    mask_end = seg["offset"] + seg["duration"] + 0.05

                    # Clamp the expanded mask so it never overwrites known transcribed speech
                    for t_seg in transcription_segs:
                        t_start = t_seg["offset"]
                        t_end = t_start + t_seg["duration"]

                        # If the mask bleeds into the end of a speech segment, push it forward
                        if mask_start < t_end and seg["offset"] >= t_end:
                            mask_start = t_end
                        # If the mask bleeds into the start of a speech segment, pull it back
                        if (
                            mask_end > t_start
                            and (seg["offset"] + seg["duration"]) <= t_start
                        ):
                            mask_end = t_start

                    if mask_end > segment_start and mask_start < segment_end:
                        m_start = int((mask_start - segment_start) * sr)
                        m_end = int((mask_end - segment_start) * sr)
                        m_start = max(0, m_start)
                        m_end = min(len(segment_audio), m_end)

                        # ... proceed with original_clip, rms_volume, and fade logic ...

                        target_len = m_end - m_start
                        if target_len <= 0:
                            continue

                        # 1. Calculate RMS before we mute anything
                        original_clip = segment_audio[m_start:m_end].copy()
                        rms_volume = (
                            np.sqrt(np.mean(original_clip**2))
                            if len(original_clip) > 0
                            else 0.05
                        )
                        # Clamp the RMS so the massive pop doesn't make our pink noise deafeningly loud
                        rms_volume = min(0.1, max(rms_volume, 0.04))

                        # 2. Define true crossfade boundaries (10ms outside the mask)
                        fade_len = int(sr * 0.01)
                        f_start = max(0, m_start - fade_len)
                        f_end = min(len(segment_audio), m_end + fade_len)

                        actual_fade_in_len = m_start - f_start
                        actual_fade_out_len = f_end - m_end

                        # 3. HARD MUTE the entire masked section to kill the transient pop instantly
                        segment_audio[m_start:m_end] = 0.0

                        # 4. Fade OUT the good audio just BEFORE the mask hits
                        if actual_fade_in_len > 0:
                            fade_out_curve = np.linspace(
                                1, 0, actual_fade_in_len
                            )
                            segment_audio[f_start:m_start] *= fade_out_curve

                        # 5. Fade IN the good audio just AFTER the mask ends
                        if actual_fade_out_len > 0:
                            fade_in_curve = np.linspace(
                                0, 1, actual_fade_out_len
                            )
                            segment_audio[m_end:f_end] *= fade_in_curve

                        # 6. Generate pink noise for the ENTIRE region (fades + mask)
                        noise_len = f_end - f_start
                        noise = generate_pink_noise(
                            noise_len, volume_scale=rms_volume * 3.0
                        )

                        # 7. Apply crossfades to the noise so it seamlessly blends
                        if actual_fade_in_len > 0:
                            noise[:actual_fade_in_len] *= np.linspace(
                                0, 1, actual_fade_in_len
                            )
                        if actual_fade_out_len > 0:
                            noise[-actual_fade_out_len:] *= np.linspace(
                                1, 0, actual_fade_out_len
                            )

                        # 8. Mix the noise into the timeline (since the masked section is 0.0, this replaces it perfectly)
                        segment_audio[f_start:f_end] += noise

            sf.write(
                out_filename, segment_audio, sr, format="FLAC", subtype="PCM_16"
            )
            output_blob.upload_from_filename(out_filename)

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "text": segment_gt_text,
                    "example_id": example_id,
                    "segment_id": segment_id,
                    "offset": segment_start,
                    "duration": segment_end - segment_start,
                }
            )

    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{full_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))
    logger.info(
        f"Pipeline Complete. Exported {len(final_manifest_entries)} segments."
    )

    if final_manifest_entries:
        df = pd.DataFrame(final_manifest_entries)
        display(df[["segment_id", "offset", "duration", "text"]].head(10))